# IA responsable con Fairlearn y Aequitas

Este tutorial muestra cómo **preparar datos, entrenar un modelo y revisar si sus errores cambian entre grupos** usando el conjunto de datos COMPAS.


In [ ]:
from pathlib import Path
import warnings

import pandas as pd
from IPython.display import display

# Librerías locales
from preprocessing import Preprocesar
from train_model import train_model
from auditar import auditar_modelo

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.3f}".format


## 1. Conocer los datos

### ¿Qué es COMPAS?

COMPAS es un sistema utilizado en Estados Unidos para estimar el riesgo de reincidencia. Aquí no intentamos copiar su funcionamiento interno. Usamos los datos publicados por ProPublica para entrenar un modelo sencillo y estudiar sus errores.

En `compas-scores-two-years.csv`, cada fila representa una persona evaluada. Las columnas pertenecen a distintos momentos:

1. **Datos personales:** edad, sexo y raza.
2. **Información conocida durante la evaluación:** antecedentes y cargo actual.
3. **Resultados de COMPAS:** `decile_score`, `score_text` y otras puntuaciones.
4. **Información posterior:** nuevas detenciones, nuevos cargos y seguimiento.
5. **Objetivo:** `two_year_recid`, que indica si se observó reincidencia durante los dos años siguientes.

> **Idea clave — fuga de información:** el modelo solo debe utilizar información disponible al hacer la predicción. Usar un dato posterior sería como intentar predecir la calificación de un examen después de verla.

### Variables utilizadas

En `variables.txt`, el prefijo `--` marca variables consideradas para el proyecto. Esto no significa que todas deban entrar al modelo. `two_year_recid` es un caso especial: es la respuesta que queremos predecir y **nunca se usa como entrada**.

| Variable | Motivo para conservarla |
| --- | --- |
| `age` | Fue una de las variables más útiles en el análisis previo. |
| `priors_count` | Resume antecedentes y fue la más importante en Random Forest. |
| `juv_fel_count` | Cuenta delitos graves juveniles anteriores. |
| `juv_misd_count` | Cuenta delitos menores juveniles anteriores. |
| `c_charge_degree` | Describe la gravedad del cargo actual. |
| `sex`, `race` | Permiten medir y mitigar diferencias entre grupos. Son atributos sensibles y deben interpretarse con cuidado. |

Se excluyen nombres e identificadores, las puntuaciones generadas por COMPAS y las variables que revelan hechos futuros. `age_cat` se conserva para la auditoría, pero no se usa junto con `age` como predictor para evitar representar dos veces casi la misma información.


In [ ]:
csv_path = Path("data") / "raw" / "compas-scores-two-years.csv"
datos = pd.read_csv(csv_path)

columnas_ejemplo = [
    "age",
    "sex",
    "race",
    "priors_count",
    "c_charge_degree",
    "two_year_recid",
]

print(f"Archivo: {csv_path.resolve()}")
print(f"Tamaño: {len(datos):,} filas × {datos.shape[1]} columnas")
display(datos[columnas_ejemplo].head())


## 2. Preparar la información

`Preprocesar` limpia las observaciones, transforma las categorías y produce dos tablas:

- **Datos del modelo:** variables numéricas y `two_year_recid`.
- **Contexto de auditoría:** identificador, etiqueta real y atributos protegidos sin transformar.

Empezaremos sin mitigación para obtener un **modelo base** fácil de interpretar. Después podremos comparar las dos alternativas de Fairlearn.


In [ ]:
# Punto de partida recomendado.
PREPROCESSING_METHOD = None

# Alternativas para comparar después:
# PREPROCESSING_METHOD = "CorrelationRemover"
# PREPROCESSING_METHOD = "PrototypeRepresentationLearner"

method_label = PREPROCESSING_METHOD or "sin_mitigacion"

compas_preprocessed, compas_audit_context = Preprocesar(
    csv_path,
    preprocessing_method=PREPROCESSING_METHOD,
)

resumen_preparacion = pd.DataFrame(
    {
        "Tabla": ["Datos del modelo", "Contexto de auditoría"],
        "Filas": [len(compas_preprocessed), len(compas_audit_context)],
        "Columnas": [compas_preprocessed.shape[1], compas_audit_context.shape[1]],
    }
)

display(resumen_preparacion)
print("Primeras filas preparadas para el modelo:")
display(compas_preprocessed.head(3))


> **Qué debemos observar:** las dos tablas deben tener el mismo número de filas. Así, cada predicción podrá relacionarse con la persona y el grupo correctos durante la auditoría.


## 3. Entrenar y evaluar el modelo

Se utiliza una regresión logística con validación cruzada estratificada de 10 particiones. En cada vuelta, el modelo se entrena con nueve partes y predice la parte restante.

De esta manera, cada persona recibe una predicción hecha por un modelo que **no utilizó esa misma observación para entrenarse**.


In [ ]:
result, audit_predictions = train_model(
    compas_preprocessed,
    compas_audit_context,
    model_name=f"LR_{method_label}",
    feature_set=method_label,
    n_splits=10,
    random_state=42,
)

nombres_metricas = {
    "accuracy": "Exactitud",
    "precision": "Precisión",
    "recall": "Sensibilidad",
    "f1_score": "F1",
}

resumen_metricas = pd.DataFrame(
    {
        "Promedio": pd.Series(result["metrics"]),
        "Variación": pd.Series(result["metrics_std"]),
    }
).rename(index=nombres_metricas).round(3)

print(f"Estrategia: {method_label}")
display(resumen_metricas)


### Cómo leer las métricas

| Métrica | Pregunta que responde |
| --- | --- |
| Exactitud | ¿Qué proporción total se clasificó correctamente? |
| Precisión | De las predicciones positivas, ¿cuántas fueron correctas? |
| Sensibilidad | De los casos positivos reales, ¿cuántos detectó el modelo? |
| F1 | ¿Qué tan equilibradas están precisión y sensibilidad? |

> Un buen resultado general **no demuestra equidad**. Dos grupos pueden obtener una exactitud parecida y, aun así, sufrir tipos de error diferentes.


## 4. Auditar los resultados con Aequitas

La tabla `audit_predictions` contiene la predicción, el resultado real y los atributos protegidos de cada observación. Aequitas la utiliza para calcular:

- **FPR:** proporción de falsos positivos;
- **FNR:** proporción de falsos negativos;
- **disparidad:** razón entre la métrica de un grupo y la del grupo de referencia;
- **paridad:** indica si la razón está dentro del intervalo aproximado de `0.80` a `1.25`.

Los grupos de referencia son `Caucasian`, `Male` y `25 - 45`. Elegir otros grupos puede cambiar la comparación.


In [ ]:
resultado = auditar_modelo(
    audit_predictions,
    tau=0.80,
    alpha=0.05,
)

columnas_absolutas = [
    "attribute_name",
    "attribute_value",
    "group_size",
    "fpr",
    "fnr",
    "precision",
]
columnas_disparidad = [
    "attribute_name",
    "attribute_value",
    "fpr_disparity",
    "fnr_disparity",
    "pprev_disparity",
]
columnas_paridad = [
    "attribute_name",
    "attribute_value",
    "FPR Parity",
    "FNR Parity",
    "Statistical Parity",
]

def columnas_disponibles(tabla, columnas):
    return [columna for columna in columnas if columna in tabla.columns]

metricas_grupo = resultado["group_metrics"].loc[
    :, columnas_disponibles(resultado["group_metrics"], columnas_absolutas)
].copy()
disparidades = resultado["disparities"].loc[
    :, columnas_disponibles(resultado["disparities"], columnas_disparidad)
].copy()
paridad_grupo = resultado["group_fairness"].loc[
    :, columnas_disponibles(resultado["group_fairness"], columnas_paridad)
].copy()


### Orden recomendado de lectura

1. Revisa `group_size`: los grupos pequeños producen resultados menos estables.
2. Compara `fpr` y `fnr`: muestran qué tipo de error recibe cada grupo.
3. Revisa las disparidades: un valor cercano a `1` indica mayor semejanza con el grupo de referencia.
4. Consulta la paridad: sirve como alerta, no como prueba definitiva de justicia o discriminación.


In [ ]:
etiquetas = {
    "attribute_name": "Atributo",
    "attribute_value": "Grupo",
    "group_size": "Personas",
    "fpr": "Falsos positivos",
    "fnr": "Falsos negativos",
    "precision": "Precisión",
    "fpr_disparity": "Disparidad FPR",
    "fnr_disparity": "Disparidad FNR",
    "pprev_disparity": "Disparidad de positivos",
}

print("Métricas por grupo:")
display(metricas_grupo.rename(columns=etiquetas).round(3))

print("Disparidades respecto al grupo de referencia:")
display(disparidades.rename(columns=etiquetas).round(3))

print("Alertas de paridad:")
display(paridad_grupo.rename(columns=etiquetas))


## 5. Revisar el resultado global

El resumen global ayuda a localizar rápidamente una alerta, pero no sustituye las tablas anteriores. La conclusión debe considerar el tamaño de los grupos, la magnitud de los errores y el contexto del problema.


In [ ]:
print("Resumen por atributo:")
display(resultado["attribute_fairness"])

print("Resumen global:")
display(resultado["overall_fairness"])

output_dir = Path("results") / method_label
output_dir.mkdir(parents=True, exist_ok=True)

metricas_grupo.to_csv(output_dir / "group_metrics.csv", index=False)
disparidades.to_csv(output_dir / "disparities.csv", index=False)
paridad_grupo.to_csv(output_dir / "group_fairness.csv", index=False)

print(f"Resultados guardados en: {output_dir.resolve()}")


## 6. Comparar estrategias

Ejecuta nuevamente el tutorial cambiando solo `PREPROCESSING_METHOD`:

| Valor | Propósito |
| --- | --- |
| `None` | Modelo base sin mitigación. |
| `"CorrelationRemover"` | Reduce correlaciones lineales con `race` y `sex`. |
| `"PrototypeRepresentationLearner"` | Aprende una representación alternativa para reducir diferencias entre grupos. |

Compara siempre dos dimensiones:

1. **Utilidad:** exactitud, precisión, sensibilidad y F1.
2. **Equidad:** FPR, FNR, disparidades y alertas de paridad.

Una estrategia puede mejorar una dimensión y empeorar otra. El objetivo no es encontrar un único número «perfecto», sino entender esos compromisos.


## 7. Conclusiones y límites

Antes de aceptar un resultado, responde:

- ¿El modelo mantiene una utilidad razonable?
- ¿Qué grupos reciben más falsos positivos o falsos negativos?
- ¿El método de mitigación mejora las disparidades sin dañar demasiado el rendimiento?
- ¿La cantidad de datos de cada grupo permite confiar en la comparación?

> **Límite importante:** estas métricas describen asociaciones en este conjunto de datos. No demuestran causalidad ni garantizan que el sistema sea justo. Una evaluación responsable también necesita revisión social, jurídica y del contexto de uso.
